In [1]:
import os

# used for configuring biogeme use of GPU, unused
# os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
# os.environ["CUDA_VISIBLE_DEVICES"] = ""

# Destination-choice + stay/move model via Larch

In [2]:
import os
import sys
from datetime import UTC, datetime
from pathlib import Path

import larch as lx
import numpy as np
import pandas as pd
from larch import PX
import xarray as xr

sys.path.insert(0, os.path.abspath(".."))
from lib import model_spec as lm
from lib import modeling_util as lut
from lib import io as lio


In [3]:
num_alternatives = 50
unixtime = int(datetime.now(UTC).timestamp())
path = "../data/estdata_10_2018_50.parquet"
data_file = Path(path).stem

### Read data

In [4]:
df_train = lio.read_estdata(
    path,
    num_alternatives,
).sample(frac=0.8, random_state=42)
print(df_train.shape)

/workspace/migration/lib/io.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["person_id"] = np.arange(len(df))
/workspace/migration/lib/io.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["ALT_CHOICE"] = 0


(202472, 1166)


In [5]:
# need to have the sentinel values be different so that SAME_CBSA works correctly
df_train["NAME_NUM.ORIG"].min(), df_train["ALT1_CBSA"].min()

(np.int64(-2), np.float32(-1.0))

### Look at collinearity in requested columns

In [ ]:
# stack all alternatives into long format
frames = []
for i in range(1, num_alternatives + 1):
    cols = {
        f"ALT{i}_{v}": v
        for v in lm.required_alt_suffixes()
        if f"ALT{i}_{v}" in df_train.columns
    }
    frames.append(df_train[list(cols)].rename(columns=cols))

long = pd.concat(frames, ignore_index=True)

corr = long.corr()

In [ ]:
c = corr.abs()
pairs = (
    c.where(np.triu(np.ones(c.shape), k=1).astype(bool))
    .stack()
    .nlargest(20)
)
print(pairs)

In [ ]:
orig_vars = [v for v in lm.required_individual_columns() if v in df_train.columns]  # skip any missing
corr = df_train[orig_vars].corr()

c = corr.abs()
pairs = (
    c.where(np.triu(np.ones(c.shape), k=1).astype(bool))
    .stack()
    .nlargest(20)
)
print(pairs.to_string())

### Reshape to long format, build the Larch dataset

`lib.util.build_long_data` builds the long `(person_id, alt)` table shared by this notebook and
`modeling_torch_choice.ipynb`: `alt=0` is staying, `alt=1..num_alternatives` are the move alternatives, with the
stay/move-context values for shared coefficients (e.g. `proportion_same_age_18_34`) written under the
same column name so they tie to one coefficient downstream, and `log_pop_offset` (destination
population on move rows, origin population on the stay row) left as an un-parameterized term. 

The returned `long_df` is already sorted by `(person_id, alt)`; `Dataset.construct.from_idca` takes it
directly (indexed by `(caseid, altid)`) -- no manual reshape into arrays needed, unlike the torch-choice
port.


In [9]:
long_df, STAY_ONLY_TERMS, SHARED_TERMS, MOVE_ONLY_TERMS = lut.build_long_data(
    df_train, num_alternatives
)
varnames = STAY_ONLY_TERMS + SHARED_TERMS + MOVE_ONLY_TERMS

long_df.set_index(["person_id", "alt"], inplace=True)

In [ ]:
lut.print_utility_formula(long_df.reset_index()[["person_id", "alt", "choice", "log_pop_offset", "sampling_correction"] + varnames])

In [11]:
ds = lx.Dataset.construct.from_idca(
    long_df[["choice", "log_pop_offset", "sampling_correction", "PWGTP"] + varnames], crack=True
)
ds

<xarray.Dataset> Size: 3GB
Dimensions:                                       (person_id: 202472, alt: 51)
Coordinates:
  * person_id                                     (person_id) int32 810kB 0 ....
  * alt                                           (alt) int32 204B 0 1 ... 49 50
Data variables: (12/64)
    choice                                        (person_id, alt) int32 41MB ...
    log_pop_offset                                (person_id, alt) float32 41MB ...
    sampling_correction                           (person_id, alt) float32 41MB ...
    PWGTP                                         (person_id) float64 2MB 1.3...
    stay                                          (person_id, alt) float32 41MB ...
    stay_age_18_22                                (person_id, alt) float32 41MB ...
    ...                                            ...
    destchoice_vacancy_rate                       (person_id, alt) float32 41MB ...
    destchoice_med_rent_k                         (person_id, alt) float32 41MB ...
    destchoice_med_earnings_10k_no_degree         (person_id, alt) float32 41MB ...
    destchoice_med_earnings_10k_degree            (person_id, alt) float32 41MB ...
    destchoice_unemp                              (person_id, alt) float32 41MB ...
    destchoice_alt_commute                        (person_id, alt) float32 41MB ...
Attributes:
    _caseid_:  person_id
    _altid_:   alt

In [12]:
del long_df, df_train

### Setting up the model

One `P(name) * X(name)` term per `varnames` entry via the `PX` shorthand, summed on a plain local
variable and assigned to `m.utility_ca` once at the end -- **not** built with `m.utility_ca += ...` in
a loop, which silently discards everything but the last term (see intro cell). No separate ASC/intercept
term is added -- `stay` in `STAY_ONLY_TERMS` is already an explicit ASC for staying, matching
`fit_intercept=False` in the torch-choice port / Biogeme not adding an implicit ASC of its own.

`log_pop_offset` gets a coefficient too, then `m.lock_value("log_pop_offset", 1)` pins it at exactly 1
(`holdfast`), reproducing Biogeme's bare `log(Variable(...))` calls (and xlogit's `addit=`) without
needing a custom subclass the way the torch-choice port did.


In [13]:
m = lx.Model(ds)
m.title = f"us_mnl_{data_file}_{unixtime}"
m.compute_engine = "numba"

# all alternatives have the same utility function
# stay-specific columns have their values zeroed out for destination alternatives and vice versa
# PX represents a column multiplied by a coefficient that will be estimated
total_utility = PX(varnames[0])
for name in varnames[1:]:
    total_utility = total_utility + PX(name)
total_utility = total_utility + PX("log_pop_offset") + PX("sampling_correction")
m.utility_ca = total_utility

m.choice_ca_var = "choice"
m.weight_co_var = "PWGTP"
# all alternatives are available for everyone
# no availability_ca_var needed.

print("null log-likelihood:", m.loglike_null())

# fix the size term coefficient, it is a constant
m.lock_value("log_pop_offset", 1)
m.lock_value("sampling_correction", 1)

m.ordering = [
    ("Stay", "stay.*"),
    ("Destination-only", "destchoice.*"),
    ("Offset", "log_pop_offset"),
]


null log-likelihood: -796084.5995089597


In [14]:
print(m.weight_co_var)

None


In [15]:
m.utility_functions()

<xmle.Elem 'div' with 1 children>

In [ ]:
# weights = lut.extract_weights("results/us_mnl_estdata_50_2018_100_1785877411_spec.yaml")
# m.pvals = weights

In [ ]:
# optional cell: turns on the nested structure

# nested logit: alt=0 (stay) stays a direct root child (== a degenerate nest fixed at 1.0);
# alts 1..num_alternatives go under a "Move" nest with an estimated logsum coefficient.
# m.graph.new_node(
#     parameter="mu_move",
#     children=list(range(1, num_alternatives + 1)),
#     name="Move",
# )
# m.set_value("mu_move", value=0.5, initvalue=0.5, minimum=0.001, maximum=1.0)

# m.title = f"us_nested_{data_file}_{unixtime}"

### Fitting

In [18]:
result = m.maximize_loglike(method="BHHH")
result


┣          loglike: np.float64(-523536.59857921465)
┣                x: destchoice_T34                       -0.884269
┃                   destchoice_alt_commute               -1.424963
┃                   destchoice_birthstate                 0.283975
┃                   destchoice_logdist                   -1.156520
┃                   destchoice_med_earnings_10k_degree    0.207333
┃                                                           ...   
┃                   stay_metro                            0.457143
┃                   stay_mil                             -1.342414
┃                   stay_recently_divorced_or_widowed    -0.621184
┃                   stay_single_parent                    0.580807
┃                   stay_vacancy_rate                    -0.345351
┃                   Length: 62, dtype: float64
┣        tolerance: np.float64(2.9354384771860516e-06)
┣            steps: array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])
┣          message: 'Optimization terminated successfully'
┣     elapsed_time: datetime.timedelta(seconds=117, microseconds=769032)
┣           method: 'BHHH'
┣          n_cases: 1012360
┣ iteration_number: 13

In [ ]:
m.calculate_parameter_covariance()
m.add_parameter_array("covariance_matrix", m.parameters["ihess"])

In [ ]:
m.calculate_parameter_covariance(robust=True)
m.parameter_summary()


In [ ]:
report = lx.Reporter(title=m.title)
report << "# Parameter Summary" << m.parameter_summary()
report << "# Estimation Statistics" << m.estimation_statistics()
report << "# Utility functions" << m.utility_functions()
report.save(
    f"results/{m.title}.html",
    overwrite=True,
    metadata=m.dumps(),
)
m.save(f"results/{m.title}_spec.yaml")